<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [ ]:
import smbus2
import time
import struct
from typing import List, Tuple, Optional

class MLX90642:
    # Constants from MLX90642.h
    DEFAULT_I2C_ADDR = 0x66
    
    # Error codes
    NACK_ERR = 1
    INVAL_VAL_ERR = 2
    TIMEOUT_ERR = 3
    
    # Status values
    YES = 1
    NO = 0
    
    # Memory addresses
    ID1_ADDR = 0x1230
    FW_VER_ADDRESS1 = 0xFFF8
    FW_VER_ADDRESS2 = 0xFFFA
    
    # Data addresses
    AUX_DATA_ADDRESS = 0x2E02
    IR_DATA_ADDRESS = 0x2E2A
    TO_DATA_ADDRESS = 0x342C
    TA_DATA_ADDRESS = 0x3A2C
    PROGRESS_DATA_ADDRESS = 0x3C10
    FLAGS_ADDRESS = 0x3C14
    
    # Configuration addresses
    REFRESH_RATE_ADDRESS = 0x11F0
    EMISSIVITY_ADDRESS = 0x11F2
    APPLICATION_CONFIG_ADDRESS = 0x11F4
    I2C_CONFIG_ADDRESS = 0x11FC
    I2C_SA_ADDRESS = 0x11FE
    REFLECTED_TEMP_ADDRESS = 0xEEEE
    
    # Masks and shifts
    FLAGS_BUSY_MASK = 0x0001
    FLAGS_READY_MASK = 0x0100
    FLAGS_READY_SHIFT = 8
    REFRESH_RATE_MASK = 0x0007
    OUTPUT_FORMAT_MASK = 0x0100
    MEAS_MODE_MASK = 0x0800
    
    # Opcodes and commands
    CONFIG_OPCODE = 0x3A2E
    CMD_OPCODE = 0x0180
    RESET_CMD = 0x0006
    START_SYNC_MEAS_CMD = 0x0001
    SLEEP_CMD = 0x0007
    
    # Timing constants
    RESET_TIME = 10  # ms
    EE_WRITE_TIME = 15  # ms
    POLL_TIME_MS = 2
    MAX_POLL_TRIES = 100
    REF_TIME = 2000
    
    # Configuration values
    CONT_MEAS_MODE = 0
    STEP_MEAS_MODE = 0x0800
    TEMPERATURE_OUTPUT = 0
    NORMALIZED_DATA_OUTPUT = 0x0100
    REF_RATE_2HZ = 2
    REF_RATE_4HZ = 3
    REF_RATE_8HZ = 4
    REF_RATE_16HZ = 5
    REF_RATE_32HZ = 6
    
    # Pixel count
    TOTAL_NUMBER_OF_PIXELS = 768
    TOTAL_NUMBER_OF_AUX = 20
    NUMBER_OF_ID_WORDS = 4
    
    def __init__(self, i2c_bus: int = 1, i2c_addr: int = DEFAULT_I2C_ADDR):
        """Initialize MLX90642 sensor."""
        self.i2c_addr = i2c_addr
        self.bus = smbus2.SMBus(i2c_bus)
    
    def __del__(self):
        """Clean up I2C bus."""
        if hasattr(self, 'bus'):
            self.bus.close()
            
    def _i2c_read(self, start_address: int, num_words: int) -> List[int]:
        """
        MLX90642 block read I2C command.
        Implements the block read protocol from the datasheet.
        """
        try:
            # Send start address (16-bit, MSB first)
            msg_write = smbus2.i2c_msg.write(self.i2c_addr, 
                                           [start_address >> 8, start_address & 0xFF])
            
            # Read data (2 bytes per word, MSB first)
            msg_read = smbus2.i2c_msg.read(self.i2c_addr, num_words * 2)
            
            self.bus.i2c_rdwr(msg_write, msg_read)
            
            # Convert bytes to 16-bit words
            data = []
            raw_data = list(msg_read)
            for i in range(0, len(raw_data), 2):
                word = (raw_data[i] << 8) | raw_data[i + 1]
                data.append(word)
            
            return data
            
        except Exception as e:
            print(f"I2C read error: {e}")
            return []
    
    def _i2c_config(self, write_address: int, data: int) -> int:
        """
        MLX90642 configuration I2C command.
        Implements configuration protocol from datasheet.
        """
        try:
            # Configuration command format: opcode + address + data
            cmd_data = [
                (self.CONFIG_OPCODE >> 8) & 0xFF,  # MSB of opcode
                self.CONFIG_OPCODE & 0xFF,          # LSB of opcode
                (write_address >> 8) & 0xFF,        # MSB of address
                write_address & 0xFF,               # LSB of address
                (data >> 8) & 0xFF,                 # MSB of data
                data & 0xFF                         # LSB of data
            ]
            
            msg = smbus2.i2c_msg.write(self.i2c_addr, cmd_data)
            self.bus.i2c_rdwr(msg)
            return 0
            
        except Exception as e:
            print(f"I2C config error: {e}")
            return -1
    
    def _i2c_command(self, command: int) -> int:
        """
        MLX90642 I2C command send.
        For reset, start/sync, and sleep commands.
        """
        try:
            cmd_data = [
                (self.CMD_OPCODE >> 8) & 0xFF,  # MSB of opcode
                self.CMD_OPCODE & 0xFF,          # LSB of opcode
                (command >> 8) & 0xFF,           # MSB of command
                command & 0xFF                   # LSB of command
            ]
            
            msg = smbus2.i2c_msg.write(self.i2c_addr, cmd_data)
            self.bus.i2c_rdwr(msg)
            return 0
            
        except Exception as e:
            print(f"I2C command error: {e}")
            return -1
    
    def _i2c_wake_up(self) -> int:
        """MLX90642 wake-up command."""
        try:
            # Wake up command is just opcode 0x57
            msg = smbus2.i2c_msg.write(self.i2c_addr, [0x57])
            self.bus.i2c_rdwr(msg)
            return 0
            
        except Exception as e:
            print(f"I2C wake up error: {e}")
            return -1
        
    def get_device_id(self) -> Optional[List[int]]:
        """Get the device ID (4 words)."""
        return self._i2c_read(self.ID1_ADDR, self.NUMBER_OF_ID_WORDS)
    
    def _wait_ms(self, milliseconds: int):
        """Wait for specified milliseconds."""
        time.sleep(milliseconds / 1000.0)
        
    def get_refresh_rate(self) -> int:
        """Get the refresh rate setting."""
        data = self._i2c_read(self.REFRESH_RATE_ADDRESS, 1)
        if not data:
            return -1
        
        rate = data[0] & self.REFRESH_RATE_MASK
        return rate if rate >= self.REF_RATE_2HZ else self.REF_RATE_2HZ
    
    def set_refresh_rate(self, rate: int) -> int:
        """Set the refresh rate."""
        if rate < self.REF_RATE_2HZ or rate > self.REF_RATE_32HZ:
            return -self.INVAL_VAL_ERR
        
        data = self._i2c_read(self.REFRESH_RATE_ADDRESS, 1)
        if not data:
            return -1
        
        new_data = (data[0] & ~self.REFRESH_RATE_MASK) | rate
        result = self._i2c_config(self.REFRESH_RATE_ADDRESS, new_data)
        self._wait_ms(self.EE_WRITE_TIME)
        return result
    
    def get_measurement_mode(self) -> int:
        """Get the measurement mode."""
        data = self._i2c_read(self.APPLICATION_CONFIG_ADDRESS, 1)
        if not data:
            return -1
        return data[0] & self.MEAS_MODE_MASK
    
    def get_output_format(self) -> int:
        """Get the output data format."""
        data = self._i2c_read(self.APPLICATION_CONFIG_ADDRESS, 1)
        if not data:
            return -1
        return data[0] & self.OUTPUT_FORMAT_MASK
    
    def set_measurement_mode(self, mode: int) -> int:
        """Set the measurement mode."""
        if mode not in [self.CONT_MEAS_MODE, self.STEP_MEAS_MODE]:
            return -self.INVAL_VAL_ERR
        
        data = self._i2c_read(self.APPLICATION_CONFIG_ADDRESS, 1)
        if not data:
            return -1
        
        new_data = (data[0] & ~self.MEAS_MODE_MASK) | mode
        result = self._i2c_config(self.APPLICATION_CONFIG_ADDRESS, new_data)
        self._wait_ms(self.EE_WRITE_TIME)
        return result
    
    def set_emissivity(self, emissivity: int) -> int:
        """Set the emissivity (scaled by 2^14, so 1.0 = 0x4000)."""
        result = self._i2c_config(self.EMISSIVITY_ADDRESS, emissivity & 0xFFFF)
        self._wait_ms(self.EE_WRITE_TIME)
        return result
    
    def get_firmware_version(self) -> Optional[Tuple[int, int, int]]:
        """Get firmware version as (major, minor, patch)."""
        data = self._i2c_read(self.FW_VER_ADDRESS1, 2)
        if len(data) != 2:
            return None
        
        # Extract version numbers according to datasheet
        major = (data[0] >> 8) & 0xFF
        minor = data[1] & 0xFF
        patch = (data[1] >> 8) & 0xFF
        
        return (major, minor, patch)
    
    def is_device_busy(self) -> int:
        """Check if device is busy."""
        data = self._i2c_read(self.FLAGS_ADDRESS, 1)
        if not data:
            return -1
        return data[0] & self.FLAGS_BUSY_MASK
    
    def is_data_ready(self) -> int:
        """Check if data is ready for read-out."""
        data = self._i2c_read(self.FLAGS_ADDRESS, 1)
        if not data:
            return -1
        return (data[0] & self.FLAGS_READY_MASK) >> self.FLAGS_READY_SHIFT
    
    def set_output_format(self, format_type: int) -> int:
        """Set the output data format."""
        if format_type not in [self.TEMPERATURE_OUTPUT, self.NORMALIZED_DATA_OUTPUT]:
            return -self.INVAL_VAL_ERR
        
        data = self._i2c_read(self.APPLICATION_CONFIG_ADDRESS, 1)
        if not data:
            return -1
        
        new_data = (data[0] & ~self.OUTPUT_FORMAT_MASK) | format_type
        result = self._i2c_config(self.APPLICATION_CONFIG_ADDRESS, new_data)
        self._wait_ms(self.EE_WRITE_TIME)
        return result   
    
    def is_read_window_open(self) -> int:
        """Check if read window is open for consistent frame data."""
        ready_status = self.is_data_ready()
        if ready_status != self.YES:
            return ready_status

        busy_status = self.is_device_busy()
        if busy_status < 0:
            return busy_status

        return self.YES if busy_status == self.NO else self.NO
    
    def clear_data_ready(self) -> int:
        """Clear the data ready flag."""
        # Reading from TO_DATA_ADDRESS clears the flag
        data = self._i2c_read(self.TO_DATA_ADDRESS, 1)
        if not data:
            return -1
        return self.is_data_ready()
    
    def get_progress(self) -> int:
        """Get measurement progress (0-100%)."""
        data = self._i2c_read(self.PROGRESS_DATA_ADDRESS, 1)
        return data[0] if data else -1

    def start_sync_measurement(self) -> int:
        """Start/sync a new measurement."""
        return self._i2c_command(self.START_SYNC_MEAS_CMD)

    def goto_sleep(self) -> int:
        """Put device in sleep mode."""
        return self._i2c_command(self.SLEEP_CMD)

    def get_refresh_time_ms(self) -> int:
        """Calculate expected refresh time based on refresh rate setting."""
        rate = self.get_refresh_rate()
        if rate < 0:
            return rate

        return self.REF_TIME >> rate

    def get_image_data(self) -> Optional[List[int]]:
        """Get the calculated temperature/normalized image data."""
        data = self._i2c_read(self.TO_DATA_ADDRESS, self.TOTAL_NUMBER_OF_PIXELS)
        return data if len(data) == self.TOTAL_NUMBER_OF_PIXELS else None

    def get_raw_ir_data(self) -> Optional[List[int]]:
        """Get the raw IR data."""
        data = self._i2c_read(self.IR_DATA_ADDRESS, self.TOTAL_NUMBER_OF_PIXELS)
        return data if len(data) == self.TOTAL_NUMBER_OF_PIXELS else None

    def get_aux_data(self) -> Optional[List[int]]:
        """Get auxiliary data."""
        data = self._i2c_read(self.AUX_DATA_ADDRESS, self.TOTAL_NUMBER_OF_AUX)
        return data if len(data) == self.TOTAL_NUMBER_OF_AUX else None

    def get_sensor_temperature(self) -> Optional[float]:
        """Get sensor temperature in Celsius."""
        data = self._i2c_read(self.TA_DATA_ADDRESS, 1)
        if not data:
            return None

        # Convert according to datasheet: value / 100
        return data[0] / 100.0

    def initialize(self) -> int:
        """Initialize the device and wait for first valid data."""
        ref_time = self.get_refresh_time_ms()
        if ref_time < 0:
            return ref_time

        # Clear any existing data ready flag
        if self.clear_data_ready() != 0:
            return -self.INVAL_VAL_ERR

        # Start a new measurement
        if self.start_sync_measurement() < 0:
            return -1

        # Wait for expected measurement time
        self._wait_ms(ref_time)

        # Poll for data ready
        for _ in range(self.MAX_POLL_TRIES):
            self._wait_ms(self.POLL_TIME_MS)
            status = self.is_data_ready()
            if status < 0:
                return status
            if status == self.YES:
                return 0

        return -self.TIMEOUT_ERR

    def measure_now(self) -> Optional[List[int]]:
        """Take a single measurement and return the image data."""
        ref_time = self.get_refresh_time_ms()
        if ref_time < 0:
            return None

        # Clear data ready flag
        if self.clear_data_ready() != 0:
            return None

        # Start measurement
        if self.start_sync_measurement() < 0:
            return None

        # Wait for measurement
        self._wait_ms(ref_time)

        # Poll for completion
        for _ in range(self.MAX_POLL_TRIES):
            self._wait_ms(self.POLL_TIME_MS)
            if self.is_data_ready() == self.YES:
                return self.get_image_data()

        return None 
    def convert_temperature_data(self, raw_data: List[int]) -> List[float]:
        """
        Convert raw temperature data to Celsius.
        According to datasheet: Temperature = value / 50.0
        """
        temperatures = []
        for value in raw_data:
            # Handle two's complement for negative values
            if value > 32767:
                value = value - 65536
            temp_celsius = value / 50.0
            temperatures.append(temp_celsius)

        return temperatures

    def get_pixel_array(self, data: List[int]) -> List[List[float]]:
        """Convert linear data to 24x32 pixel array (24 rows, 32 columns)."""
        if len(data) != self.TOTAL_NUMBER_OF_PIXELS:
            return []

        pixel_array = []
        for row in range(24):
            pixel_row = []
            for col in range(32):
                index = row * 32 + col
                pixel_row.append(data[index])
            pixel_array.append(pixel_row)

        return pixel_array